# Predicting House Sale Prices - Ames, Iowa Housing Data

**Author:** Majid M. D Ben Ghet
**Date: ** September 15th 2026

This notebook works through a full data science project: taking a real, messy
real estate dataset, applying varous method of data cleaning, exploring it and drawing conclusions, and building a number of prediction models to
estimate a house's sale price from its characteristics.

## 1. Business Understanding

A real estate analytics team wants a way to estimate a **fair listing price** for a
house before it goes on the market, based on things like its size, condition,
location and age. A model like this would help agents:

- price new listings competitively instead of guessing,
- flag listings that look significantly over- or under-priced compared to similar
  houses,
- give buyers a quick "is this a fair price" sanity check.

**Data source:** the dataset covers 1,460 residential home sales in Ames, Iowa, with
about 80 characteristics recorded for each house (size, quality ratings, garage,
basement, neighborhood, year built, and so on) along with the final sale price.

## 2. Setup
We will first start by importing the relevant libraries to our project.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

# make sure the output folders exist before we try to save anything into them
if not os.path.exists("../figures"):
    os.makedirs("../figures")

if not os.path.exists("../models"):
    os.makedirs("../models")

# fixing the random seed so the notebook gives the same results every time it is run
RANDOM_STATE = 42

## 3. Data Understanding

Before doing anything else, we take a first look at the data: how big it is, what
types of columns we have, and where the gaps are.

In [ ]:
df.info()

In [ ]:
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]
missing_counts = missing_counts.sort_values(ascending=False)

missing_percent = (missing_counts / len(df)) * 100

print("Columns with missing values:", len(missing_counts))
for column_name in missing_counts.index:
    count = missing_counts[column_name]
    percent = missing_percent[column_name]
    print(column_name, "-", count, "missing (", round(percent, 1), "% )")

In [ ]:
plt.figure(figsize=(9, 8))
plt.barh(missing_counts.index, missing_counts.values, color="steelblue")
plt.gca().invert_yaxis()
plt.xlabel("Number of missing values")
plt.title("Missing values per column (before cleaning)")
plt.tight_layout()
plt.savefig("../figures/fig_missingness.png", dpi=150)
plt.show()

In [ ]:
number_of_duplicate_rows = df.duplicated().sum()
print("Number of duplicate rows:", number_of_duplicate_rows)

print()
print("Summary of the target variable, SalePrice:")
print(df["SalePrice"].describe())